In [11]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [18]:
df = pd.read_pickle("../../data/processed/17_cleaned_master_data.pkl")

# Behandlung des Features "team_class"
## Grundidee:
Die aktuellen Einträge in der Spalte "team_class" sind folgende:

In [4]:
df["team_class"].value_counts()

team_class
WT      129813
PT       31820
PCT      24822
PRT       9555
CLUB        38
Name: count, dtype: int64

### Problem 1: 

PT und WT stehen beide für die höchste Wertung eines Teams. Die WorldTour wurde 2009 eingeführt. In 2009 und 2010 (Übergangsphase) existieren beide Wertungen im Datensatz



In [5]:
counts = (
    df.loc[df["team_class"].isin(["WT", "PT"])]
    .groupby(["year", "team_class"])
    .size()
    .reset_index(name="count")
    .sort_values(["year", "team_class"])
)

counts

,year,team_class,count
0,2005,PT,7725
1,2006,PT,8253
2,2007,PT,7933
3,2008,PT,7034
4,2009,PT,839
5,2009,WT,6264
6,2010,PT,36
7,2010,WT,7372
8,2011,WT,8020
9,2012,WT,8141


#### Lösung:
WT und PT zusammenführen

### Problem 2:
PCT & PRT sind beides Einträge für Continental-Pro-Teams. PCT Einträge gibt es bis 2019. Danah gibt es sauber getrennt nur PRT Einträge

In [6]:
counts = (
    df.loc[df["team_class"].isin(["PCT", "PRT"])]
    .groupby(["year", "team_class"])
    .size()
    .reset_index(name="count")
    .sort_values(["year", "team_class"])
)

counts

,year,team_class,count
0,2005,PCT,974
1,2006,PCT,445
2,2007,PCT,972
3,2008,PCT,1496
4,2009,PCT,2115
5,2010,PCT,2269
6,2011,PCT,1932
7,2012,PCT,1817
8,2013,PCT,1479
9,2014,PCT,1856


#### Lösung:
PCT und PRT zusammenführen

### Problem 3:
Eine ordinale Skala (3,2,1) ist unnwillkürlich. Es würde suggerieren, das die Abstände zwischen allen Teamklassen gleich groß sind.

#### Lösung:
Elite = WT + PT ; 
Continental = PCT + PRT ; 
other = CLUB


### Problem 4: Korrelation mit year und Fahrerstärke
team_class ist ein Zeitmarker dafür, wann WT eingeführt wurde und wann PCT --> PRT. 

In [14]:
tier_map = {
    "WT": "elite",
    "PT": "elite",
    "PCT": "continental",
    "PRT": "continental",
    "CLUB": "other",
}
df["team_tier"] = df["team_class"].map(tier_map)

# Plausibilitätscheck
assert df["team_tier"].isna().sum() == 0, "Unmapped team_class values"
df["team_tier"].value_counts()

team_tier
elite          161633
continental     34377
other              38
Name: count, dtype: int64

In [8]:
year_tier = pd.crosstab(
    df["year"],
    df["team_tier"],
    normalize="index"   # Anteil pro Jahr
).round(3)

year_tier

team_tier,continental,elite,other
year,,,
2005,0.112,0.888,0.000
2006,0.051,0.949,0.000
2007,0.109,0.891,0.000
2008,0.175,0.825,0.000
2009,0.229,0.771,0.000
2010,0.234,0.766,0.000
2011,0.194,0.806,0.000
2012,0.182,0.818,0.000
2013,0.151,0.849,0.000


Cramér's V (Stärke der Assoziation von year <-> team_tier): Hoher Wert = starke Kopplung

In [9]:
from scipy.stats import chi2_contingency

ct = pd.crosstab(df["year"], df["team_tier"])
chi2, p, dof, expected = chi2_contingency(ct)

n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"Cramér's V (year vs team_tier): {cramers_v:.3f}")

Cramér's V (year vs team_tier): 0.082


Zusammenhang team_tier und Fahrerstärke

In [10]:
df.groupby("team_tier")[["rider_points_season", "rider_rank_season"]].agg(
    ["count", "median", "mean"]
).round(1)

rider_points_season               rider_rank_season              
                          count median   mean             count median   mean
team_tier                                                                    
continental               34377  146.0  228.8             34377  409.0  498.4
elite                    161633  221.0  374.5            161633  272.0  364.2
other                        38  344.0  344.0                38  291.0  291.0

#### Lösung:
team_tier behalten, da es Überlappungen zwischen elite und continental gibt. (continental mean ≈ elite median). team_tier kann über die individuelle Saisonstärke hinaus feine Unterschiede (z.B. Rolle im Team) abbilden. Dennoch liegt eine Informationsredundanz zwischen team_tier und rider_points_season / rider_rank_season vor. Es ist also kein unabhängiger Qualitätsindikator.

## Implementierung & Checkpoint

**Input:**  `17_cleaned_master_data.pkl`  
**Output:** `18_cleaned_master_data.pkl`  

**Änderungen:**
- Neue Spalte `team_tier` (kategorial: `elite`, `continental`, `other`)
- Spalte `team_class` entfernt
- Mapping: WT/PT → elite, PCT/PRT → continental, CLUB → other

In [15]:
# team_class entfernen
df = df.drop(columns=["team_class"])

# Optional: kategorischen Datentyp für späteres EBM/ML
df["team_tier"] = df["team_tier"].astype("category")

# Kurz prüfen
assert "team_class" not in df.columns
assert "team_tier" in df.columns
print(f"Shape: {df.shape[0]:,} × {df.shape[1]}")
print(df["team_tier"].dtype)
df[["current_team", "team_tier"]].head()

Shape: 196,048 × 48
category


,current_team,team_tier
0,Quickstep - Innergetic,elite
1,Crédit Agricole,elite
2,Davitamon - Lotto,elite
3,"Cofidis, le Crédit par Téléphone",elite
4,Liquigas,elite


In [16]:
# Speichern (Konvention wie in 07-02, 07-06, …)
pfad = "../../data/processed/18_cleaned_master_data.pkl"
df.to_pickle(pfad)
print(f"Gespeichert: {pfad}")

Gespeichert: ../../data/processed/18_cleaned_master_data.pkl


In [17]:
# Reload-Check (Roundtrip)
df_check = pd.read_pickle(pfad)
assert "team_class" not in df_check.columns
assert set(df_check["team_tier"].cat.categories) == {"elite", "continental", "other"}
print("Roundtrip OK")

Roundtrip OK
